In [ ]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Upload kaggle.json
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"rajan1999","key":"f81a67dde166587f5cc8ce53caa18315"}'}

In [ ]:
# Configure Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download dataset
!kaggle datasets download -d rajan1999/movies

Dataset URL: https://www.kaggle.com/datasets/rajan1999/movies
License(s): CC0-1.0
100% 137M/137M [00:05<00:00, 25.1MB/s]



In [ ]:
# Extract files
!unzip -o movies.zip

Archive:  movies.zip
  inflating: movies.csv              


In [ ]:
movies = pd.read_csv("/content/movies.csv")

In [ ]:
movies.sample(5)

,id,title,details_raw,credits_raw,keywords
23700,9838,King Solomon's Mines,"{'adult': False, 'backdrop_path': '/pqWTlRp8m6...","{'id': 9838, 'cast': [{'adult': False, 'gender...","{'id': 9838, 'keywords': [{'id': 1454, 'name':..."
23334,237756,Kill Me Three Times,"{'adult': False, 'backdrop_path': '/fyFxUv5teT...","{'id': 237756, 'cast': [{'adult': False, 'gend...","{'id': 237756, 'keywords': [{'id': 782, 'name'..."
2549,1516737,Shticky Fingers,"{'adult': False, 'backdrop_path': '/s2DgtrHMgP...","{'id': 1516737, 'cast': [{'adult': False, 'gen...","{'id': 1516737, 'keywords': []}"
14177,324786,Hacksaw Ridge,"{'adult': False, 'backdrop_path': '/vDKRMZGFTK...","{'id': 324786, 'cast': [{'adult': False, 'gend...","{'id': 324786, 'keywords': [{'id': 233, 'name'..."
6403,454640,The Angry Birds Movie 2,"{'adult': False, 'backdrop_path': '/tntH2nSeaG...","{'id': 454640, 'cast': [{'adult': False, 'gend...","{'id': 454640, 'keywords': [{'id': 2041, 'name..."


In [ ]:
movies.shape

(24050, 5)

##**Data Cleaning**

In [ ]:
import ast

# Convert these stringified columns back to Python dicts
movies["details_raw"] = movies["details_raw"].apply(ast.literal_eval)
movies["credits_raw"] = movies["credits_raw"].apply(ast.literal_eval)
movies["keywords"] = movies["keywords"].apply(ast.literal_eval)

In [ ]:
Movies = pd.DataFrame({
    "id": movies["id"],
    "original_language": movies["details_raw"].apply(lambda x: x.get("original_language")),
    "title": movies["title"],
    "original_title": movies["details_raw"].apply(lambda x: x.get("original_title")),
    "overview": movies["details_raw"].apply(lambda x: x.get("overview")),
    "genres": movies["details_raw"].apply(lambda x: x.get("genres")),
    "keywords": movies["keywords"],
    "cast": movies["credits_raw"].apply(lambda x: x.get("cast")),
    "crew": movies["credits_raw"].apply(lambda x: x.get("crew"))
})

In [ ]:
Movies.sample(5)

,id,original_language,title,original_title,overview,genres,keywords,cast,crew
18029,14367,en,Adventures in Babysitting,Adventures in Babysitting,"When plans with her boyfriend fall through, hi...","[{'id': 35, 'name': 'Comedy'}]","{'id': 14367, 'keywords': [{'id': 2604, 'name'...","[{'adult': False, 'gender': 1, 'id': 1951, 'kn...","[{'adult': False, 'gender': 1, 'id': 2162, 'kn..."
21203,9872,en,Explorers,Explorers,Middle schooler Ben spends his free time watch...,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...","{'id': 9872, 'keywords': [{'id': 1612, 'name':...","[{'adult': False, 'gender': 2, 'id': 569, 'kno...","[{'adult': False, 'gender': 1, 'id': 16157, 'k..."
20650,9333,en,Last Man Standing,Last Man Standing,John Smith is a mysterious stranger who is dra...,"[{'id': 80, 'name': 'Crime'}, {'id': 28, 'name...","{'id': 9333, 'keywords': [{'id': 947, 'name': ...","[{'adult': False, 'gender': 2, 'id': 62, 'know...","[{'adult': False, 'gender': 2, 'id': 1723, 'kn..."
947,884003,te,Police Vari Heccharika,పోలీసు వరి హేచారికా,"Amid a wave of killings, officer Kanchana unco...","[{'id': 28, 'name': 'Action'}, {'id': 10749, '...","{'id': 884003, 'keywords': []}","[{'adult': False, 'gender': 0, 'id': 5528290, ...","[{'adult': False, 'gender': 0, 'id': 2499482, ..."
5498,1319969,en,Sketch,Sketch,When a young girl’s sketchbook falls into a st...,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","{'id': 1319969, 'keywords': [{'id': 268132, 'n...","[{'adult': False, 'gender': 2, 'id': 25147, 'k...","[{'adult': False, 'gender': 2, 'id': 4266328, ..."


In [ ]:
# Function to extract genres, keywords, top 3 cast, and crew(director)
def convert_genres(x):

    L = []

    for i in x:
        L.append(i["name"])
    return L

def convert_keywords(x):

  L = []

  for i in x.get("keywords", []):
    L.append(i["name"])
  return L

def convert_top3(x):
  L = []

  counter = 0

  for i in x:
    if counter != 3:
      L.append(i["name"])
      counter += 1
    else:
      break
  return L

def fetch_director(x):

  L = []

  for i in x:
    if i["job"] == "Director":
      L.append(i["name"])
      break
  return L

In [ ]:
# Extract genres, keywords, top 3 cast, and crew(director)
Movies["genres"] = Movies["genres"].apply(convert_genres)
Movies["keywords"] = Movies["keywords"].apply(convert_keywords)
Movies["cast"] = Movies["cast"].apply(convert_top3)
Movies["crew"] = Movies["crew"].apply(fetch_director)

In [ ]:
Movies.sample(5)

,id,original_language,title,original_title,overview,genres,keywords,cast,crew
10701,862551,en,Me Time,Me Time,"With his family away, a devoted stay-at-home d...",[Comedy],"[stay-at-home dad, buddy comedy]","[Kevin Hart, Mark Wahlberg, Regina Hall]",[John Hamburg]
21064,9896,en,Rat Race,Rat Race,"In an ensemble film about easy money, greed, m...","[Adventure, Comedy]","[casino, road trip, millionaire, eccentric mil...","[Rowan Atkinson, Lanei Chapman, John Cleese]",[Jerry Zucker]
20673,993784,en,Lisa Frankenstein,Lisa Frankenstein,"In 1989, a misunderstood teenager has a high s...","[Horror, Comedy, Romance]","[high school, cemetery, dark comedy, coming of...","[Kathryn Newton, Cole Sprouse, Liza Soberano]",[Zelda Williams]
21938,76640,en,The Last Stand,The Last Stand,Ray Owens is sheriff of the quiet US border to...,"[Action, Crime, Thriller]","[small town, sheriff, prisoner, escape, hostag...","[Arnold Schwarzenegger, Johnny Knoxville, Jaim...",[Kim Jee-woon]
10612,758336,en,Love Again,Love Again,"Mira Ray, dealing with the loss of her fiancé,...","[Romance, Drama, Comedy]","[new york city, journalist, based on novel or ...","[Priyanka Chopra Jonas, Sam Heughan, Céline Dion]",[Jim Strouse]


In [ ]:
# Any Missing Values?
Movies.isnull().sum()

,0
id,0
original_language,0
title,0
original_title,0
overview,0
genres,0
keywords,0
cast,0
crew,0


In [ ]:
# Drop rows where overview, genres, keywords, cast, and crew are all empty
rows_to_drop = (
    (Movies["overview"].isna() | (Movies["overview"] == "")) &
    (Movies["genres"].apply(lambda x: len(x) == 0)) &
    (Movies["keywords"].apply(lambda x: len(x) == 0)) &
    (Movies["cast"].apply(lambda x: len(x) == 0)) &
    (Movies["crew"].apply(lambda x: len(x) == 0))
)

Movies[rows_to_drop] = Movies[rows_to_drop].replace("", np.nan)
Movies.dropna(inplace = True)

In [ ]:
Movies.sample(5)

,id,original_language,title,original_title,overview,genres,keywords,cast,crew
966,1336738,id,Tabayyun,Tabayyun,"Zalina and Arlo, two individuals from differen...",[Drama],"[muslim, based on novel or book, love, single ...","[Titi Kamal, Ibrahim Risyad, Naysila Mirdad]",[Key Mangunsong]
4289,928334,pt,Julia's Lover,O Amante de Júlia,Paralyzed from the waist down after an acciden...,"[Romance, Drama]",[],"[Bianca Bin, Romulo Estrela, Sérgio Guizé]",[Vinicius Coimbra]
4320,1413643,es,Sin él,Sin él,,"[Thriller, Horror, Drama]",[],"[Aleida Torrent, David Marcé, Aida Flix]",[Emilio Martinez-Borso]
16762,138,en,Dracula,Dracula,British estate agent Renfield travels to Trans...,[Horror],"[monster, based on novel or book, transylvania...","[Bela Lugosi, Helen Chandler, David Manners]",[Tod Browning]
6176,1088514,es,The Room Next Door,La habitación de al lado,Ingrid and Martha were close friends in their ...,[Drama],"[new york city, suicide, based on novel or boo...","[Julianne Moore, Tilda Swinton, John Turturro]",[Pedro Almodóvar]


In [ ]:
# Any Duplicates?
Movies.duplicated(subset = "id").sum()

np.int64(6888)

In [ ]:
# Drop Duplicates
Movies = Movies.drop_duplicates(subset = "id", keep = "first")

In [ ]:
import requests
from time import sleep

API_KEY = "b8cd668a03ee6296f94919995b52d187"
BASE_URL = "https://api.themoviedb.org/3/movie/"

def has_english_translation(movie_id):
  """Check if the movie has an English translation on TMDB."""
  url = f"{BASE_URL}{movie_id}/translations?api_key={API_KEY}"
  try:
    resp = requests.get(url)
    data = resp.json()
    translations = data.get("translations", [])
    for t in translations:
      if t.get("iso_639_1") == "en":
        overview = t.get("data", {}).get("overview", "").strip()
        title = t.get("data", {}).get("title", "").strip()
        # Consider it translated if either title or overview has text
        if overview or title:
          return True
    return False
  except Exception as e:
    print(f"Error fetching {movie_id}: {e}")
    return None

Movies["has_en_translation"] = Movies["id"].apply(has_english_translation)

# Sleep between requests to avoid hitting rate limits
sleep(0.25)  # TMDB allows ~40 requests per 10 seconds per IP

# Filter those without English translation
no_english = Movies[
    (Movies["has_en_translation"] == False) &
    (Movies["original_language"] != "en")
    ]

print(f"Movies without English translations: {len(no_english)}")

Movies without English translations: 504


In [ ]:
no_english.head(5)

,id,original_language,title,original_title,overview,genres,keywords,cast,crew,has_en_translation
173,1514770,ja,ドジで、タメ口で、すぐ不貞腐れるけど なんだかんだ性処理ご奉仕だけは絶対に手を抜かない 正直...,ドジで、タメ口で、すぐ不貞腐れるけど なんだかんだ性処理ご奉仕だけは絶対に手を抜かない 正直...,,[],"[メイド, 主観, 中出し, 淫語, vr専用, ツンデレ]",[Arata Arina],[ZAMPA],False
337,1529517,es,Reinas de la Noche,Reinas de la Noche,,[Comedy],[],"[Catherine Siachoque, Cynthia Klitbo, Anna Cep...",[Yahayra Garrido],False
345,1497279,zh,非人哉：限时玩家,非人哉：限时玩家,,"[Drama, Animation, Comedy]",[],"[Yang Ning, Baomu Zhongyang, Su Shangqing]",[Aishan Yu],False
400,1517911,nl,21 juli 2025: Belgian Party,21 juli 2025: Belgian Party,,[Music],[],"[Francisco Schuster, Youssef Swatt’s, Colt]",[],False
428,1526724,tl,Maalikaya,Maalikaya,"Behind prison walls, inmate Kara explores her ...","[Drama, Romance]",[softcore],"[Jenn Rosa, Aliya Raymundo]",[Roman Perez Jr.],False


In [ ]:
!pip install deep_translator
!pip install fasttext
! wget https://dl.fbaipublicfiles.com/fasttext/supervised-models/lid.176.ftz

In [ ]:
import fasttext
import re
from deep_translator import GoogleTranslator

# Load language detection model
ft_model = fasttext.load_model("lid.176.ftz")

# Languages to detect
foreign_langs = {
    "zh", "ru", "ja", "ko", "ar", "he", "hi", "th", "ta", "te", "bn",
    "es", "it", "pl", "cs", "pt", "fr", "de", "tr", "nl", "sv", "da", "no"}

# Converts lists to text
def to_text(v):
  if isinstance(v, list):
    return " ".join(map(str, v))
  return str(v).strip() if v else ""

# Detect if the text looks foreign(by language or by script)
def is_foreign(text):
  text = to_text(text)
  if not text or len(text) < 3:
    return False

  # Detect non-ASCII characters
  if re.search(r"[^\x00-\x7F]", text):
    return True

  # Detect using FastText
  try:
    ft_lang = ft_model.predict(text, k = 1)[0][0].replace("__label__", "")
  except Exception:
    ft_lang = "unknown"

  # Return true if the language is in foreign set
  return ft_lang in foreign_langs

# Translate safely to English only when foreign
def translate(text):
  text = to_text(text)
  if not text or not is_foreign(text):
    return text  # Skip if empty or English
  try:
    return GoogleTranslator(source = "auto", target = "en").translate(text)
  except Exception:
    return text

# Apply to the DataFrame
def translate_df(df, columns):
  df = df.copy()
  for col in columns:
    print(f"Translating column: {col}")
    df[col] = df[col].apply(translate)
  return df

In [ ]:
columns = ["title", "overview", "keywords"]
translated_movies = translate_df(no_english, columns)

Translating column: title
Translating column: overview
Translating column: keywords


In [ ]:
no_english.head(5)

,id,original_language,title,original_title,overview,genres,keywords,cast,crew,has_en_translation
173,1514770,ja,ドジで、タメ口で、すぐ不貞腐れるけど なんだかんだ性処理ご奉仕だけは絶対に手を抜かない 正直...,ドジで、タメ口で、すぐ不貞腐れるけど なんだかんだ性処理ご奉仕だけは絶対に手を抜かない 正直...,,[],"[メイド, 主観, 中出し, 淫語, vr専用, ツンデレ]",[Arata Arina],[ZAMPA],False
337,1529517,es,Reinas de la Noche,Reinas de la Noche,,[Comedy],[],"[Catherine Siachoque, Cynthia Klitbo, Anna Cep...",[Yahayra Garrido],False
345,1497279,zh,非人哉：限时玩家,非人哉：限时玩家,,"[Drama, Animation, Comedy]",[],"[Yang Ning, Baomu Zhongyang, Su Shangqing]",[Aishan Yu],False
400,1517911,nl,21 juli 2025: Belgian Party,21 juli 2025: Belgian Party,,[Music],[],"[Francisco Schuster, Youssef Swatt’s, Colt]",[],False
428,1526724,tl,Maalikaya,Maalikaya,"Behind prison walls, inmate Kara explores her ...","[Drama, Romance]",[softcore],"[Jenn Rosa, Aliya Raymundo]",[Roman Perez Jr.],False


In [ ]:
translated_movies.head(5)

,id,original_language,title,original_title,overview,genres,keywords,cast,crew,has_en_translation
173,1514770,ja,"She's clumsy, talkative, and easily becomes un...",ドジで、タメ口で、すぐ不貞腐れるけど なんだかんだ性処理ご奉仕だけは絶対に手を抜かない 正直...,,[],Maid POV Creampie Dirty Talk VR Only Tsundere,[Arata Arina],[ZAMPA],False
337,1529517,es,Reinas de la Noche,Reinas de la Noche,,[Comedy],,"[Catherine Siachoque, Cynthia Klitbo, Anna Cep...",[Yahayra Garrido],False
345,1497279,zh,Inhumane: Limited Time Player,非人哉：限时玩家,,"[Drama, Animation, Comedy]",,"[Yang Ning, Baomu Zhongyang, Su Shangqing]",[Aishan Yu],False
400,1517911,nl,21 juli 2025: Belgian Party,21 juli 2025: Belgian Party,,[Music],,"[Francisco Schuster, Youssef Swatt’s, Colt]",[],False
428,1526724,tl,Maalikaya,Maalikaya,"Behind prison walls, inmate Kara explores her ...","[Drama, Romance]",softcore,"[Jenn Rosa, Aliya Raymundo]",[Roman Perez Jr.],False


In [ ]:
# Set "id" as index to align rows
Movies.set_index("id", inplace = True)
translated_movies.set_index("id", inplace = True)

# Update Movies with translations
Movies.update(translated_movies)

# Reset index if needed
Movies.reset_index(inplace = True)

In [ ]:
Movies.sample(5)

,id,original_language,title,original_title,overview,genres,keywords,cast,crew,has_en_translation
15753,41479,en,The Joneses,The Joneses,A seemingly perfect family moves into a suburb...,"[Comedy, Drama]","[materialism, duringcreditsstinger]","[David Duchovny, Demi Moore, Amber Heard]",[Derrick Borte],True
8791,560527,ko,The Dude in Me,내안의 그놈,"Dong-hyun is a high school student. One day, h...","[Fantasy, Comedy, Action]",[body-swap],"[Jung Jin-young, Park Sung-woong, Ra Mi-ran]",[Kang Hyo-jin],True
10980,182560,en,Dark Places,Dark Places,A woman who survived the brutal killing of her...,"[Thriller, Mystery, Drama, Crime]","[prison, sibling relationship, based on novel ...","[Charlize Theron, Nicholas Hoult, Chloë Grace ...",[Gilles Paquet-Brenner],True
15222,582913,en,The Room,The Room,Kate and Matt discover that a part of their ho...,"[Horror, Drama, Mystery, Science Fiction]","[wish, miscarriage, family, reality vs fantasy...","[Olga Kurylenko, Kevin Janssens, Francis Chapman]",[Christian Volckman],True
1566,1447771,en,The Big Sea,The Big Sea,Surfing is killing it. This $10 billion global...,[],[],[],[Lewis Arnold],True


In [ ]:
# Converting Overview from string to list
Movies["overview"] = Movies["overview"].apply(lambda x: x.split())

In [ ]:
# Remove spaces between words(genres, keywords, cast, crew)
Movies["genres"] = Movies["genres"].apply(lambda x: [i.replace(" ", "") for i in x])
Movies["keywords"] = Movies["keywords"].apply(lambda x: [i.replace(" ", "") for i in x])
Movies["cast"] = Movies["cast"].apply(lambda x: [i.replace(" ", "") for i in x])
Movies["crew"] = Movies["crew"].apply(lambda x: [i.replace(" ", "") for i in x])

In [ ]:
# Create a new column tags, a combination of overview, genres, keywords, cast, and crew
Movies["tags"] = Movies["overview"] + Movies["genres"] + Movies["keywords"] + Movies["cast"] + Movies["crew"]

In [ ]:
# Convert the tags column to a string
Movies["tags"] = Movies["tags"].apply(lambda x: " ".join(x))

In [ ]:
movies_df = Movies[["id", "title", "tags"]].copy()

In [ ]:
movies_df.sample(10)

,id,title,tags
16099,10093,The Return,Joanna Mills has a successful career but feels...
198,1359607,Miss Kobayashi's Dragon Maid: A Lonely Dragon ...,"Miss Kobayashi, Tohru, Kanna, and Iruru's peac..."
1293,1485959,Ikigake no sora,Drama TakahiroMiura MisakiHattori Nahana Shini...
3765,1480749,Time's Arrow,"A tentative teenager, entering her formative y..."
14320,11617,Rio Grande,Lt. Col. Kirby Yorke is posted on the Texas fr...
5069,206647,Spectre,A cryptic message from Bond’s past sends him o...
688,1486092,Gandhi Was Silent on Saturdays,The life of 16-year-old Mot is thrown into dis...
16377,471507,Destroyer,"When Erin Bell was a young cop, she was given ..."
8463,296989,The Other Side of The SEX,A documentary that shines a spotlight on the u...
13756,369523,The Tale,An investigation into one woman’s memory as sh...


##**Data Preprocessing**

In [ ]:
import re

In [ ]:
# Basic cleaning: remove extra spaces, line breaks, trailing spaces
def clean_text(text):
  text = re.sub(r"\s+", " ", text)   # replace multiple spaces with one
  text = text.strip()                # remove leading/trailing spaces
  return text

In [ ]:
# Apply cleaning to your movie tags column
movies_df["tags"] = movies_df["tags"].apply(clean_text)

##**Modeling**

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 23.2 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
# Load a small, fast embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert all movie tags into 384-dimensional embeddings (vectors)
embeddings = model.encode(
    movies_df["tags"].tolist(),
    show_progress_bar = True,
    convert_to_numpy = True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/536 [00:00<?, ?it/s]

In [ ]:
# Get the size of each embedding vector
d = embeddings.shape[1]

# Create a FAISS index that uses L2 distance for similarity search
index = faiss.IndexFlatL2(d)

# Add all movie embeddings to the FAISS index
index.add(embeddings)

In [ ]:
# Recommend k similar movies using FAISS search
def recommend(movie, k = 5):
    movie_index = movies_df[movies_df["title"] == movie].index[0]

    # Get the embedding of the selected movie
    query_vec = embeddings[movie_index].reshape(1,-1)

    # Search FAISS for the k nearest neighbours
    distances, indices = index.search(query_vec, k + 1)   # +1 because the closest movie to a movie is itself

    # Skip the first result because it's itself
    recommended_indices = indices[0][1:]

    # Print the recommended movie titles
    for idx in recommended_indices:
        print(movies_df.iloc[idx].title)

In [ ]:
recommend("One Piece Film: GOLD")

One Piece Film: Strong World
One Piece "3D2Y": Overcome Ace's Death! Luffy's Vow to his Friends
One Piece: The Movie
One Piece Film: Z
One Piece: Heart of Gold


In [ ]:
import pickle

In [ ]:
# Save the movies dataframe as a dict
movies_dict = movies_df.to_dict()

pickle.dump(movies_dict, open("movies_dict.pkl", "wb"))

In [ ]:
# Save embeddings
with open("embeddings.pkl", "wb") as f:
    pickle.dump(embeddings, f)

In [ ]:
# Save FAISS index
faiss.write_index(index, "faiss_index.bin")

In [ ]:
from google.colab import files
import json

# Upload the notebook you downloaded from Colab
uploaded = files.upload()

input_file = next(iter(uploaded))
output_file = input_file.replace(".ipynb", "-Github.ipynb")

with open(input_file, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove broken widget metadata
nb.get("metadata", {}).pop("widgets", None)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(nb, f)

print(f"Created: {output_file}")

# Download cleaned notebook
files.download(output_file)

Saving Movie_Recommender_System.ipynb to Movie_Recommender_System.ipynb
Created: Movie_Recommender_System-Github.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>